# Italy Audi Q4 e-tron — Preprocessing & Feature Engineering

Reads the merged Italy primary + secondary snapshot CSV from `../Scrape/outputs/`,
applies the same feature engineering as the Germany preprocessing notebook,
and outputs `outputs/audi_q4_it_snapshot_clean.parquet` + `.csv`.

**Note:** Italy is a single one-time snapshot — there are no lifecycle columns
(first_seen, last_seen, sold_timestamp, status, duration_days, sold_flag).
The output is a cross-sectional listing dataset only.

Run after `../Scrape/02_secondary_scraper_italy.ipynb`.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

SCRAPE_DIR = Path("../Scrape/outputs")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Load the most recent Italy secondary CSV (merged primary + secondary)
_secondary_csvs = sorted(
    [p for p in SCRAPE_DIR.glob("scrape_audi_q4_it_*_secondary.csv")],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not _secondary_csvs:
    raise FileNotFoundError(
        "No Italy secondary CSV found in ../Scrape/outputs/. "
        "Run the two scraper notebooks first."
    )

INPUT_CSV = _secondary_csvs[0]
print(f"Loading: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)
print(f"Shape: {df.shape}")
df.head(3)

Loading: ../Scrape/outputs/scrape_audi_q4_it_20260522_secondary.csv
Shape: (227, 57)


,listing_id,title,subtitle,variant,model_group,transmission,price,currency,price_superscript,is_conditional_price,...,exterior_color,paint_type,interior_color,upholstery_material,battery_ownership,battery_charging_time,electric_range,electric_range_city,secondary_scrape_status,secondary_scrape_error
0,8d7209f6-394c-454f-a474-b673e6502204,Audi Q4 e-tron 45 s line edition quattro 265cv,45 s line edition quattro 265cv,Q4 e-tron 45,Q4 e-tron,Automatic,"36,900",€,NaN,False,...,Black,Metallic,Black,Alcantara,NaN,NaN,NaN,NaN,ok,NaN
1,384baec8-934a-47d7-ae1c-656829b6e5d5,Audi Q4 e-tron Q4 sportback e-tron 40 s line e...,Q4 sportback e-tron 40 s line edition,Q4 e-tron Sportback,Q4 e-tron,Automatic,"35,900",€,1.0,False,...,White,Others,Black,Other,NaN,NaN,NaN,NaN,ok,NaN
2,48e267d1-5179-43cb-8876-0b527f813493,Audi Q4 e-tron 35 business advanced,35 business advanced,Q4 e-tron 35,Q4 e-tron,Automatic,"39,900",€,1.0,False,...,Grey,Others,Black,Other,NaN,NaN,NaN,NaN,ok,NaN


In [2]:
# Quick column profile
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "unique_count": df.nunique(),
    "example": df.apply(lambda col: col.dropna().iloc[0] if col.dropna().size > 0 else np.nan),
}).sort_values("missing_pct", ascending=False)

profile

,dtype,missing_count,missing_pct,unique_count,example
secondary_scrape_error,float64,227,100.0,0,NaN
electric_range_city,float64,227,100.0,0,NaN
vat_deductible,float64,227,100.0,0,NaN
delivery_possible,float64,227,100.0,0,NaN
damage_conditions,object,226,99.6,1,Not suitable for riding
battery_charging_time,object,224,98.7,3,29 min
battery_ownership,object,222,97.8,1,Included
has_360_image,object,217,95.6,1,True
raw_card_text,object,207,91.2,19,Audi Q4 e-tron 45 s line edition quattro 265cv...
electric_range,object,144,63.4,29,494 km


In [3]:
# Parsing helpers (identical to Germany preprocessing)
def parse_number(series):
    return pd.to_numeric(
        series.astype("string")
        .str.extract(r"([0-9][0-9,.]*)", expand=False)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

def parse_decimal(series):
    return pd.to_numeric(
        series.astype("string")
        .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False),
        errors="coerce"
    )

def parse_bool(series):
    return series.astype("string").str.lower().map({"true": True, "false": False}).astype("boolean")

In [4]:
# --- Numeric parsing ---
clean = df.copy()

clean["price_eur"]                = parse_number(clean["price"])
clean["mileage_km"]               = parse_number(clean["mileage"])
clean["electric_range_km"]        = parse_number(clean["electric_range"])
clean["electric_range_city_km"]   = parse_number(clean["electric_range_city"])
clean["battery_charging_time_min"]= parse_number(clean["battery_charging_time"])
clean["warranty_months"]          = parse_number(clean["warranty_text"])

is_kwh = clean["wltp_consumption"].astype("string").str.contains("kWh/100 km", na=False)
clean["wltp_consumption_kwh_100km"] = np.where(
    is_kwh, parse_decimal(clean["wltp_consumption"]), np.nan
)
clean["wltp_co2_g_km"] = parse_number(clean["wltp_co2_emissions"])

print("Numeric columns parsed.")
clean[["price_eur", "mileage_km", "electric_range_km", "power_kw", "power_hp"]].describe()

Numeric columns parsed.


,price_eur,mileage_km,electric_range_km,power_kw,power_hp
count,227.0,197.0,83.0,227.000000,227.000000
mean,41646.048458,31456.208122,449.939759,101.638767,138.224670
std,10732.893264,29274.381501,49.816088,46.310723,63.054603
min,22500.0,0.0,307.0,70.000000,95.000000
25%,32925.0,5655.0,446.5,70.000000,95.000000
50%,41500.0,25560.0,462.0,77.000000,105.000000
75%,49900.0,47431.0,487.5,125.000000,170.000000
max,73776.0,128000.0,516.0,250.000000,340.000000


In [5]:
# --- Date parsing & vehicle age ---
# Italy is a single snapshot; use the scrape date as reference
SCRAPE_DATE_STR = INPUT_CSV.stem.split("_")[4]  # e.g. '20260522' from scrape_audi_q4_it_20260522_secondary
try:
    reference_date = pd.Timestamp(SCRAPE_DATE_STR)
except Exception:
    reference_date = pd.Timestamp.today()
print(f"Reference date for vehicle age: {reference_date.date()}")

clean["first_registration_dt"] = pd.to_datetime(
    "01/" + clean["first_registration"].astype("string"),
    format="%d/%m/%Y",
    errors="coerce",
)

clean["vehicle_age_months"] = (
    (reference_date.year  - clean["first_registration_dt"].dt.year)  * 12
    + (reference_date.month - clean["first_registration_dt"].dt.month)
).clip(lower=0)

print(f"vehicle_age_months: median={clean['vehicle_age_months'].median():.0f} months, "
      f"missing={clean['vehicle_age_months'].isna().sum()}")

Reference date for vehicle age: 2026-05-22
vehicle_age_months: median=30 months, missing=35


In [6]:
# --- Boolean columns ---
bool_cols = [
    "is_conditional_price", "available_now", "warranty_exists",
    "has_full_service_history", "had_accident", "has_360_image", "delivery_possible",
]
for col in bool_cols:
    if col in clean.columns:
        clean[col + "_clean"] = parse_bool(clean[col])

print("Boolean columns parsed.")

Boolean columns parsed.


In [7]:
# --- Text-based binary flags ---
text = (
    clean["title"].fillna("") + " " +
    clean["subtitle"].fillna("")
).str.lower()

clean["is_sportback"] = clean["variant"].eq("Q4 e-tron Sportback")
clean["is_quattro"]   = text.str.contains("quattro", regex=False)
clean["is_s_line"]    = text.str.contains("s line|s-line", regex=True)
clean["has_matrix"]   = text.str.contains("matrix", regex=False)
clean["has_pano"]     = text.str.contains("pano|panorama", regex=True)
clean["has_ahk"]      = text.str.contains("ahk", regex=False)
clean["has_hud"]      = text.str.contains("hud|head-up|head up", regex=True)
clean["has_acc"]      = text.str.contains(r"\bacc\b", regex=True)
clean["has_camera"]   = text.str.contains("kamera|camera|rfk", regex=True)

flag_cols = ["is_sportback", "is_quattro", "is_s_line", "has_matrix",
             "has_pano", "has_ahk", "has_hud", "has_acc", "has_camera"]
print("Feature flag penetration (Italy):")
print(clean[flag_cols].mean().sort_values(ascending=False).map("{:.1%}".format))

Feature flag penetration (Italy):
is_s_line       40.5%
is_sportback    36.1%
is_quattro      21.1%
has_camera       1.3%
has_matrix       0.9%
has_acc          0.9%
has_pano         0.0%
has_ahk          0.0%
has_hud          0.0%
dtype: object


In [8]:
# --- Model number extraction ---
clean["model_number_v2"] = text.str.extract(
    r"(?:q4\s*)?(35|40|45|50|55)\s*(?:e[- ]?tron|quattro|qu\.|\b)",
    expand=False,
)

# Fallback: simpler pattern if the above misses some
_fallback = text.str.extract(r"\b(35|40|45|50|55)\b", expand=False)
clean["model_number_v2"] = clean["model_number_v2"].fillna(_fallback)

print("Model mix (Italy):")
print(clean["model_number_v2"].value_counts(dropna=False))

Model mix (Italy):
model_number_v2
45     85
40     76
35     33
50     13
NaN    11
55      9
Name: count, dtype: int64


In [9]:
# --- Ratio / derived features ---
clean["price_per_kw"]       = clean["price_eur"]       / pd.to_numeric(clean["power_kw"],  errors="coerce")
clean["price_per_hp"]       = clean["price_eur"]       / pd.to_numeric(clean["power_hp"],  errors="coerce")
clean["price_per_range_km"] = clean["price_eur"]       / clean["electric_range_km"]
clean["mileage_per_month"]  = clean["mileage_km"]      / clean["vehicle_age_months"].replace(0, np.nan)

# Indicator flags
clean["seller_has_rating"]  = clean["seller_rating_stars"].notna().astype(int)
clean["has_warranty_months"]= clean["warranty_months"].notna().astype(int)
clean["has_battery_info"]   = clean["battery_ownership"].notna().astype(int)
clean["has_city_range"]     = clean["electric_range_city_km"].notna().astype(int)
clean["has_charging_time"]  = clean["battery_charging_time_min"].notna().astype(int)
clean["duplicate_listing_id"] = clean["listing_id"].duplicated(keep=False)

print("Ratio and indicator features computed.")

Ratio and indicator features computed.


In [10]:
# --- Outlier nulling (same thresholds as Germany) ---
before_price  = clean["price_eur"].notna().sum()
before_range  = clean["electric_range_km"].notna().sum()
before_wltp   = clean["wltp_consumption_kwh_100km"].notna().sum()

clean.loc[clean["wltp_consumption_kwh_100km"] > 40,  "wltp_consumption_kwh_100km"] = np.nan
clean.loc[clean["wltp_co2_g_km"]              > 0,   "wltp_co2_g_km"]              = np.nan
clean.loc[clean["electric_range_km"]          < 250, "electric_range_km"]          = np.nan
clean.loc[clean["price_eur"]                  > 90_000, "price_eur"]               = np.nan

print(f"Rows nulled: price>{90_000} → {before_price - clean['price_eur'].notna().sum()} rows")
print(f"Rows nulled: range<250km → {before_range - clean['electric_range_km'].notna().sum()} rows")
print(f"Rows nulled: WLTP>40kWh → {before_wltp - clean['wltp_consumption_kwh_100km'].notna().sum()} rows")

Rows nulled: price>90000 → 0 rows
Rows nulled: range<250km → 0 rows
Rows nulled: WLTP>40kWh → 7 rows


In [11]:
# --- Basic data quality checks ---
checks = {
    "total_listings":    len(clean),
    "price_missing":     clean["price_eur"].isna().sum(),
    "mileage_missing":   clean["mileage_km"].isna().sum(),
    "age_missing":       clean["vehicle_age_months"].isna().sum(),
    "model_num_missing": clean["model_number_v2"].isna().sum(),
    "scrape_errors":     (clean["secondary_scrape_status"] == "error").sum(),
}
for k, v in checks.items():
    print(f"  {k}: {v}")

print(f"\nPrice range: €{clean['price_eur'].min():,.0f} – €{clean['price_eur'].max():,.0f}")
print(f"Median price: €{clean['price_eur'].median():,.0f}")
print(f"Median mileage: {clean['mileage_km'].median():,.0f} km")

  total_listings: 227
  price_missing: 0
  mileage_missing: 30
  age_missing: 35
  model_num_missing: 11
  scrape_errors: 0

Price range: €22,500 – €73,776
Median price: €41,500
Median mileage: 25,560 km


In [12]:
# --- Save snapshot-clean dataset ---
clean.to_csv(OUTPUT_DIR / "audi_q4_it_snapshot_clean.csv",     index=False)
clean.to_parquet(OUTPUT_DIR / "audi_q4_it_snapshot_clean.parquet", index=False)
profile.to_csv(OUTPUT_DIR / "audi_q4_it_column_profile.csv")

print(f"\nSaved: audi_q4_it_snapshot_clean  ({clean.shape[0]} rows, {clean.shape[1]} cols)")
print(f"Output dir: {OUTPUT_DIR.resolve()}")

# Print final column summary
ml_cols = [
    "price_eur", "mileage_km", "vehicle_age_months", "power_kw", "power_hp",
    "electric_range_km", "model_number_v2", "variant", "seller_type",
    "is_sportback", "is_quattro", "is_s_line", "has_matrix",
    "seller_has_rating", "has_warranty_months",
]
clean[ml_cols].describe(include="all")


Saved: audi_q4_it_snapshot_clean  (227 rows, 94 cols)
Output dir: /Users/vakof/Desktop/HOD_car/HOD_car/Italy/Preprocessing/outputs


,price_eur,mileage_km,vehicle_age_months,power_kw,power_hp,electric_range_km,model_number_v2,variant,seller_type,is_sportback,is_quattro,is_s_line,has_matrix,seller_has_rating,has_warranty_months
count,227.0,197.0,192.000000,227.000000,227.000000,83.0,216,200,227,227,227,227,227,227.000000,227.000000
unique,<NA>,<NA>,NaN,NaN,NaN,<NA>,5,4,2,2,2,2,2,NaN,NaN
top,<NA>,<NA>,NaN,NaN,NaN,<NA>,45,Q4 e-tron Sportback,Dealer,False,False,False,False,NaN,NaN
freq,<NA>,<NA>,NaN,NaN,NaN,<NA>,85,82,194,145,179,135,225,NaN,NaN
mean,41646.048458,31456.208122,29.718750,101.638767,138.224670,449.939759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.850220,0.488987
std,10732.893264,29274.381501,14.925436,46.310723,63.054603,49.816088,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.357644,0.500983
min,22500.0,0.0,1.000000,70.000000,95.000000,307.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
25%,32925.0,5655.0,16.000000,70.000000,95.000000,446.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000
50%,41500.0,25560.0,29.500000,77.000000,105.000000,462.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000
75%,49900.0,47431.0,42.000000,125.000000,170.000000,487.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
